This notebook generates the dataset used for the ExplainableAI (XAI) analysis (Figs. 3D,E). 

Specifically, we use DeepLIFT ([A. Shrikumar *et al*, 2017](https://arxiv.org/abs/1704.02685)) and provide code for:

```
- generation of attribution scores for trained neural nets (CNN & MLP)
- formatting scores to input into downstream plotting functions
```

**Note**: As of Aug 2022, Google Colab no longer supports Tensorflow 1. For this reason we cloned DeepLIFT (from here) and converted the code to be Tensorflow 2 compatible.

## 0. Mount Drive, set the main path, and import libraries

**Note**: when you execute this block you may notice a pip dependency error -- plese ignore it and proceed to execute the rest of the code

In [ ]:
#mount to drive
from google.colab import drive
try:
  drive.mount('/content/drive', force_remount=True)
except:
  drive._mount('/content/drive', force_remount=True)

# construct the path to use
import os
 
# get the current working directory
dir_path = os.path.dirname(os.path.realpath('__file__'))

# loop to crawl over your Drive and construct the correct path
for root, dirs, files in os.walk(dir_path):
    for file in files:
 
        # do not change the extension *.ipynb*
        if file.endswith('1_kmers.ipynb'):
          # check that we're using the correct parent directory
          if f'{root}/{file}'[-24:] == '/seq2yield/1_kmers.ipynb':
            # create path to the import directory
            drive_dir = f"{f'{root}/{file}'[:-13]}to_import/"

# import libraries
#%tensorflow_version 1.x
import tensorflow as tf

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from keras.models import load_model

import sys
sys.path.insert(0, drive_dir)

# builds and installs the Tensorflow V2 compatible version of DeepLIFT
!pip install --editable $drive_dir/deeplift/

# import modules
import utils, cnn_conv2d, xai_helper

!pip install 'h5py==2.10.0' --force-reinstall

## 2. Load models
**Note**: I provide MLPs and CNNs for all mutational series. Read the comments for details on how to load them.

In [ ]:
# Specify the desired input mutational series to DeepLIFT in a list format

series_lst = [13]
algo = 'CNN' #change to 'MLP' if you want to check attr_scores for multilayer perceptron

####
iter_ID, train_sz = 1, 0.75
model_ID = f'model_{series_lst}_{train_sz}_{algo}.h5'
model_PATH = f'{drive_dir}_saved/iteration_1/models/{model_ID}'

#load model in .h5 format here
model = xai_helper.load_model(model_PATH,
                              custom_objects={'r_square':xai_helper.r_square})

print(f'\nLoaded model is: {model_ID}')

## 3. Load dataset & static test sets

In [ ]:
# load .csv into a DataFrame
pd.options.display.max_rows = 10
raw_data = pd.read_csv(f'{drive_dir}Ecoli_data.csv', index_col= 0).dropna().reset_index(drop = True)

###
series_dict = utils.split_seeds(raw_data)

working_set, heldout_set = utils.get_split(series_dict = series_dict, 
                                           series_list = series_lst,
                                           load_saved = True,
                                           path = f'{drive_dir}_saved/iteration_{iter_ID}/')

print(f'Working set contains {working_set.shape[0]} sequences for train and validation\nHeldout contains {heldout_set.shape[0]} sequences for testing')

# use this to check attribution scores for NNs trained on individual series
large_test_static = heldout_set[heldout_set.seed == series_lst[0]].reset_index(drop=True)

## 4. Run DeepLIFT and return scores in a Pandas dataframe

The *scores_df* contains:
```
- 96 columns where each cell contains the calculated attribution scores for each nucleotide in a sequence
- meas_protein, describing the measured protein expression
- pred_protein, describing the predicted protein expression
```

In [ ]:
tf.compat.v1.disable_eager_execution() #important to disable Tensorflow's eager execution!

scores_df = xai_helper.build_attr_matrix(series_lst = series_lst,
                                         test_master = large_test_static,
                                         iter_ID = 1, 
                                         train_sz = 0.75, 
                                         algo = 'CNN', 
                                         deepLIFT_option = 'default',
                                         drive_dir_master = f'{drive_dir}_saved/'
                                         )

# display the resulting dataframe
display(scores_df)